# EXP06: 出场深度分析

**合并**: NB14 + NB15 + NB17

1. TP 扫描 (8 变体)
2. BUY vs SELL Trailing 差异分析
3. 持仓时间分箱精细分析

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Baseline: {len(trades)} trades, PnL=${sum(t.pnl for t in trades):.0f}")

## Part 1: TP 扫描

In [ ]:
tp_results = []
for tp in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]:
    r = run_variant(data, f"TP={tp}", **{**LIVE_KWARGS, "tp_atr_mult": tp})
    r['tp_mult'] = tp
    tp_results.append(r)

print_ranked(tp_results)

In [ ]:
for r in tp_results:
    t_list = r['_trades']
    tp_exits = sum(1 for t in t_list if t.exit_reason == 'TP')
    trail_exits = sum(1 for t in t_list if t.exit_reason == 'Trailing')
    n = len(t_list)
    print(f"TP={r['tp_mult']}: {tp_exits}/{n} TP exits ({tp_exits/n*100:.1f}%), "
          f"{trail_exits}/{n} Trailing ({trail_exits/n*100:.1f}%)")

## Part 2: BUY vs SELL Trailing 差异

In [ ]:
df = pd.DataFrame([{
    'direction': t.direction, 'strategy': t.strategy,
    'exit_reason': t.exit_reason.split(':')[0] if ':' in t.exit_reason else t.exit_reason,
    'pnl': t.pnl, 'bars_held': t.bars_held, 'win': 1 if t.pnl > 0 else 0,
} for t in trades])

print("=== Trailing Stop by Direction ===")
for d in ['BUY', 'SELL']:
    subset = df[df['direction'] == d]
    trailing = subset[subset['exit_reason'] == 'Trailing']
    non_trailing = subset[subset['exit_reason'] != 'Trailing']
    print(f"\n  {d}: {len(subset)} trades total")
    if len(trailing) > 0:
        print(f"    Trailing: {len(trailing)} ({len(trailing)/len(subset)*100:.1f}%) "
              f"avg PnL=${trailing['pnl'].mean():.2f}, WR={trailing['win'].mean()*100:.1f}%, "
              f"median bars={trailing['bars_held'].median():.0f}")
    print(f"    Non-trailing: {len(non_trailing)} ({len(non_trailing)/len(subset)*100:.1f}%) "
          f"avg PnL=${non_trailing['pnl'].mean():.2f}")

In [ ]:
print("\n=== Exit Reason x Direction ===")
pivot = df.pivot_table(index='exit_reason', columns='direction',
                       values='pnl', aggfunc=['sum', 'count', 'mean']).round(2)
print(pivot)

print("\n=== Keltner Only: Exit x Direction ===")
kc = df[df['strategy'] == 'keltner']
print(kc.groupby(['direction', 'exit_reason']).agg(
    count=('pnl', 'count'), total_pnl=('pnl', 'sum'),
    avg_pnl=('pnl', 'mean'), avg_bars=('bars_held', 'mean'),
).round(2))

## Part 3: 持仓时间分箱分析

In [ ]:
df['hours_held'] = df['bars_held'] * 0.25
bins = [0, 4, 8, 12, 16, 20, 24, 32, 40, 48, 60, 999]
labels = ['1-4', '5-8', '9-12', '13-16', '17-20', '21-24', '25-32', '33-40', '41-48', '49-60', '60+']
df['bin'] = pd.cut(df['bars_held'], bins=bins, labels=labels)

print("=== PnL by Hold Time Bin (M15 bars) ===")
summary = df.groupby('bin', observed=True).agg(
    count=('pnl', 'count'), total_pnl=('pnl', 'sum'), avg_pnl=('pnl', 'mean'),
    win_rate=('win', 'mean'), avg_hours=('hours_held', 'mean'),
).round(2)
print(summary)

In [ ]:
print("\n=== Cumulative PnL if max_hold = N bars ===")
for max_bars in [4, 8, 12, 16, 20, 24, 32, 40, 48, 60]:
    within = df[df['bars_held'] <= max_bars]['pnl'].sum()
    beyond = df[df['bars_held'] > max_bars]['pnl'].sum()
    n_within = len(df[df['bars_held'] <= max_bars])
    n_beyond = len(df[df['bars_held'] > max_bars])
    hours = max_bars * 0.25
    print(f"  max_hold={max_bars:3d} ({hours:5.1f}h): within=${within:8.0f} ({n_within}), "
          f"beyond=${beyond:8.0f} ({n_beyond})")

In [ ]:
for strat in ['keltner', 'orb']:
    sdf = df[df['strategy'] == strat]
    if len(sdf) == 0:
        continue
    print(f"\n=== {strat} Hold Time Distribution ===")
    s = sdf.groupby('bin', observed=True).agg(
        count=('pnl', 'count'), total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'), win_rate=('win', 'mean'),
    ).round(2)
    print(s)